In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

from FEX.utils import fex
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
from data.generate_data import make_static_sf_adjacency
timesteps=5000
adj_matrix = make_static_sf_adjacency(100, 500, gamma_in=3.5, gamma_out=3.5)
print(torch.mean(adj_matrix.sum(dim=1))) # number of incoming edges (degree)


In [ ]:
import matplotlib.pyplot as plt
from data.generate_data import make_timeseries
timeseries, t_derivs = make_timeseries(num_samples=timesteps, adjacency=adj_matrix, snr=None)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
node=3
ax.plot(timeseries[:, node, 0].cpu(), timeseries[:, node, 1].cpu(), timeseries[:, node, 2].cpu(), label='True')
# ax.plot(timeseries2[:, node, 0].cpu(), timeseries2[:, node, 1].cpu(), timeseries2[:, node, 2].cpu(), label='True 2')
ax.legend()
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

In [ ]:
# combine the 2 trajectories
# timeseries = torch.cat([timeseries, timeseries2], dim=0)
# t_derivs = torch.cat([t_derivs, t_derivs2], dim=0)
dimx_fex = fex.CoupledFEX('depth_3_leaves_4_config', 'depth_2_tree_config', 0, controller_epochs=400, controller_lr=0.003, finetune_epochs=20000, finetune_lr=1e-4, num_fex_epochs=120, self_lr=0.02, inter_lr=0.02, bfgs_epochs=0, bfgs_lr=0.65, poolsize=10, device=device, expression_threshold=0.1)
dimx_fex.fit(timeseries, t_derivs, adj_matrix, num_workers=5, finetune_bs=64)

In [ ]:
print(f"dimx_fex: {dimx_fex}")

In [ ]:
dimy_fex = fex.SingleFEX('depth_2_tree_config', 1, num_finetune_epochs=5000, controller_epochs=200, num_fex_epochs=60, device=device, expression_threshold=0.01)
dimy_fex.fit(timeseries, t_derivs, num_workers=5)

In [ ]:
dimz_fex = fex.SingleFEX('depth_2_tree_config', 2, controller_lr=0.01, controller_epochs=250, num_finetune_epochs=10000, finetune_lr=0.002, self_lr=0.04, num_fex_epochs=80, device=device, expression_threshold=0.001)
dimz_fex.fit(timeseries, t_derivs, num_workers=5)

In [ ]:
print(f"dimx_fex: {dimx_fex}")
print(f"dimy_fex: {dimy_fex}")
print(f"dimz_fex: {dimz_fex}")

In [ ]:
predicted_states = torch.zeros(timesteps + 1, timeseries.size(1), timeseries.size(2), device=device)
predicted_states[0] = timeseries[0]
dt = 0.01
with torch.no_grad():
    for t in range(timesteps):
        state = predicted_states[t]

        dx_dt = torch.cat([
            dimx_fex.predict(state, adj_matrix.to(device)), # dimx_fex.predict(state, adj_matrix),
            dimy_fex.predict(state),
            dimz_fex.predict(state)
        ], dim=-1)

        predicted_states[t+1] = state + dt * dx_dt

        if not torch.isfinite(predicted_states[t+1]).all():
            print(f"Non-finite state at timestep {t + 1}")
            break
        
from FEX.utils.plots import plot_dynamics

node = 80
fig = plot_dynamics(timeseries[:, node, 0].cpu(), timeseries[:, node, 1].cpu(), timeseries[:, node, 2].cpu(), predicted_states[:, node, :].cpu(), elev=15, azim=75)
fig.show()
